# SoccerMon: Colab audit and GitHub sync
Run in order. Only the small subjective archive is downloaded initially.

In [ ]:
# CELL 1 — replace with your repository URL after creating the GitHub repository.
REPO_URL = 'PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE'
BRANCH = 'main'
DOWNLOAD_OBJECTIVE = False
OBJECTIVE_FILE = 'objective-TeamA-2020.zip'

In [ ]:
# CELL 2 — mount persistent Drive storage.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, zipfile, requests, pandas as pd
DRIVE_ROOT = Path('/content/drive/MyDrive/athlete-risk')
RAW_DIR, PROCESSED_DIR = DRIVE_ROOT / 'raw', DRIVE_ROOT / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Data location:', DRIVE_ROOT)

In [ ]:
# CELL 3 — clone your code repository (no data is cloned).
if REPO_URL.startswith('PASTE_'):
    raise ValueError('Set REPO_URL in CELL 1.')
REPO_DIR = Path('/content/Athlete-Injury-Prediction')
if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
%cd /content/Athlete-Injury-Prediction
!git status --short

In [ ]:
# CELL 4 — retrieve the official Zenodo file list and download only subjective.zip.
record = requests.get('https://zenodo.org/api/records/10033832', timeout=30).json()
files = pd.DataFrame([{'name': f['key'], 'size_gb': round(f['size']/1024**3, 3), 'url': f['links']['content']} for f in record['files']])
display(files[['name', 'size_gb']])
def download_zenodo_file(name):
    target = RAW_DIR / name
    if target.exists(): return target
    url = files.set_index('name').loc[name, 'url']
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(target, 'wb') as handle:
            for chunk in response.iter_content(1024 * 1024):
                if chunk: handle.write(chunk)
    return target
subjective_zip = download_zenodo_file('subjective.zip')
if DOWNLOAD_OBJECTIVE: objective_zip = download_zenodo_file(OBJECTIVE_FILE)

In [ ]:
# CELL 5 — extract only subjective.zip and list source CSV files.
subjective_dir = RAW_DIR / 'subjective'
if not subjective_dir.exists():
    with zipfile.ZipFile(subjective_zip) as archive: archive.extractall(subjective_dir)
csv_files = list(subjective_dir.rglob('*.csv'))
print(*[str(p.relative_to(subjective_dir)) for p in csv_files], sep='\n')

In [ ]:
# CELL 6 — create a schema-only report: no player rows or measurements are included.
report = {'source': 'SoccerMon Zenodo 10033832', 'files': []}
for csv_path in csv_files:
    frame = pd.read_csv(csv_path, nrows=100)
    report['files'].append({'path': str(csv_path.relative_to(subjective_dir)), 'columns': frame.columns.tolist(), 'inferred_types': {c: str(t) for c, t in frame.dtypes.items()}, 'rows_sampled': len(frame)})
report_path = REPO_DIR / 'data' / 'reports' / 'soccermon_schema_report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(report_path.read_text()[:5000])

In [ ]:
# CELL 7 — download this tiny report and upload it to this chat.
from google.colab import files as colab_files
colab_files.download(str(report_path))

## Optional: commit only the schema report
Create a GitHub fine-grained personal access token limited to this repository, then store it as a Colab secret named GITHUB_TOKEN. Never paste a token in this notebook or chat.

In [ ]:
# CELL 8 — optional GitHub push. Only the schema report is staged.
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
if not token: raise ValueError('Add GITHUB_TOKEN as a Colab secret or skip this cell.')
%cd /content/Athlete-Injury-Prediction
!git config user.name 'Colab Schema Report'
!git config user.email 'colab@users.noreply.github.com'
!git add data/reports/soccermon_schema_report.json
!git commit -m 'docs: add SoccerMon schema report'
push_url = REPO_URL.replace('https://', f'https://{token}@')
!git push $push_url HEAD:$BRANCH
del token, push_url